# Hugging Face Transformers — inferencia directa

Hasta ahora hemos usado modelos a través de APIs (Anthropic, Ollama).
Con Transformers accedemos al modelo directamente en Python —
sin servidor intermedio, sin HTTP, el modelo corre en el mismo proceso.

Es más complejo pero te da control total:
ves exactamente qué entra, qué sale y cómo funciona por dentro.

## Qué veremos
- Cargar un modelo pequeño desde HF directamente en Python
- Entender qué es un tokenizer y cómo convierte texto en tokens
- Hacer inferencia y comparar con Ollama

In [ ]:
from dotenv import load_dotenv
load_dotenv()
from transformers import AutoTokenizer

# Cargamos solo el tokenizer de un modelo pequeño
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

texto = "Hola, soy un desarrollador aprendiendo IA"

# Tokenizamos
tokens = tokenizer(texto)
print("IDs de tokens:", tokens["input_ids"])
print("Número de tokens:", len(tokens["input_ids"]))

# Decodificamos de vuelta a texto
decoded = tokenizer.decode(tokens["input_ids"])
print("Decodificado:", decoded)

# Ver cada token individualmente
for id in tokens["input_ids"]:
    print(f"  {id} → '{tokenizer.decode([id])}'")

## Pipeline — inferencia de alto nivel

`pipeline` es la abstracción más simple de Transformers.
Envuelve tokenizer + modelo + decodificación en una sola llamada.

Es el equivalente a Ollama pero dentro de Python — sin servidor intermedio.
Más adelante bajaremos un nivel y usaremos `AutoModelForCausalLM` directamente
para tener control total sobre cada paso.

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="distilgpt2")

resultado = generator(
    "La inteligencia artificial es",
    max_new_tokens=50,
    temperature=0.7,
    do_sample=True
)

print(resultado[0]["generated_text"])

## El idioma importa — distilgpt2 es un modelo inglés

distilgpt2 fue entrenado principalmente en inglés.
En español genera texto incoherente porque no conoce el idioma.
En inglés la calidad mejora notablemente — aunque sigue siendo un modelo pequeño.
Esto ilustra por qué el idioma de entrenamiento es crítico al elegir un modelo.

In [ ]:
resultado_en = generator(
    "Artificial intelligence is",
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
    pad_token_id=50256  # eos_token_id de GPT2
)

print(resultado_en[0]["generated_text"])

## Modelo multilingüe — mismo código, mejor modelo

distilgpt2 solo funciona bien en inglés porque fue entrenado en inglés.
Cambiando el modelo en una sola línea tenemos acceso a uno multilingüe
que entiende español y sigue instrucciones.

Esto es la potencia de Transformers: la interfaz es siempre la misma

In [ ]:
from transformers import pipeline

generator_es = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    pad_token_id=0
)

resultado = generator_es(
    "La inteligencia artificial es",
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
)

print(resultado[0]["generated_text"])

## AutoModelForCausalLM — control total

`pipeline` abstrae tokenizer + modelo + decodificación.
Aquí lo hacemos paso a paso para entender exactamente qué ocurre:

1. Tokenizamos el texto manualmente
2. Se lo pasamos al modelo
3. El modelo devuelve IDs de tokens
4. Decodificamos los IDs de vuelta a texto

Es más verbose pero ves exactamente qué entra y qué sale en cada paso.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Paso 1: texto → tokens
texto = "La inteligencia artificial es"
inputs = tokenizer(texto, return_tensors="pt")
print("Tokens IDs:", inputs["input_ids"])

# Paso 2: el modelo genera nuevos tokens
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

# Paso 3: tokens → texto
respuesta = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\nRespuesta:", respuesta)